<a href="https://colab.research.google.com/github/ronhasgreyrat/summer-internship/blob/main/FakeNewsDetector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

LOAD DATA


In [ ]:
!pip install gradio -q

import pandas as pd
import numpy as np
import re
import string
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import gradio as gr

CLEAN TEXT

In [ ]:
fake_df = pd.read_csv('False.csv')
true_df = pd.read_csv('Truth.csv')

fake_df['label'] = 0
true_df['label'] = 1

df = pd.concat([fake_df, true_df], axis=0)
df = df.sample(frac=1).reset_index(drop=True)

print(df.shape)
df.head()

SPLIT + VECTORIZE

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>+', '', text)
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub(r'\n', '', text)
    text = re.sub(r'\w*\d\w*', '', text)
    return text

df['text'] = df['title'] + " " + df['text']
df['text'] = df['text'].apply(clean_text)

TRAIN + EVALUATE

In [ ]:
X = df['text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

vectorizer = TfidfVectorizer(stop_words='english', max_df=0.7)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

Accuracy: 98.55%
              precision    recall  f1-score   support

        Fake       0.99      0.98      0.99      4669
        Real       0.98      0.99      0.98      4311

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980



TEST YOUR OWN HEADLINE

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

INTERACTIVE NEWS

In [ ]:
joblib.dump(model, 'fake_news_model.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')
print("Backend saved.")

FAKE (71.5% confidence)
FAKE (85.1% confidence)


In [ ]:
def predict_news(text):
    if not text.strip():
        return "Please enter some text.", ""

    cleaned = clean_text(text)
    vec = vectorizer.transform([cleaned])
    pred = model.predict(vec)[0]
    prob = model.predict_proba(vec)[0]

    label = "🟢 REAL NEWS" if pred == 1 else "🔴 FAKE NEWS"
    confidence = prob[pred] * 100

    return label, f"{confidence:.2f}%"

In [ ]:
with gr.Blocks(title="Fake News Detector") as demo:
    gr.Markdown("# 📰 Fake News Detector")
    gr.Markdown("Paste a news article or headline below to check if it's real or fake.")

    with gr.Row():
        with gr.Column():
            input_text = gr.Textbox(
                label="News Text",
                placeholder="Paste news article or headline here...",
                lines=8
            )
            submit_btn = gr.Button("Analyze", variant="primary")

        with gr.Column():
            output_label = gr.Textbox(label="Prediction")
            output_confidence = gr.Textbox(label="Confidence")

    submit_btn.click(
        fn=predict_news,
        inputs=input_text,
        outputs=[output_label, output_confidence]
    )

    input_text.submit(
        fn=predict_news,
        inputs=input_text,
        outputs=[output_label, output_confidence]
    )

demo.launch(debug=True)